<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
StateGraph Basics
</b></font> </br></p>

---

**Beitrag zum Leitprojekt:** Der Briefing-State ist der Arbeitszustand des Meeting- & Research-Briefing-Agent innerhalb eines Laufs — **Planen** heißt hier: Analyse, Entwurf und Qualitätsprüfung greifen auf dieselbe, gemeinsam geteilte Wahrheitsquelle zu, statt Ergebnisse nur lose durchzureichen.

In [ ]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith Env-Vars VOR allen LangChain-Imports setzen
import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"]    = "M09-StateGraph-Basics"
os.environ["LANGSMITH_ENDPOINT"]   = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace
)

setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import BASELINE, ROUTER, JUDGE, PLANNER, WORKER, WORKER_PREMIUM, CODING, EMBEDDINGS
# LangSmith Tracing
run_cfg = {
    "run_name": "M09_StateGraph_Basics",
    "tags": ["m09", "stategraph"],
    "metadata": {"notebook": "M09", "version": "1.0"}
}


**Was passiert hier?**

1. `StateGraph(...)` — definiert den Graphen mit dem State-Schema

# 1 | Übersicht
---



*Warum LangGraph?* hat gezeigt, **warum** LangGraph existiert und einen ersten 1-Node-Graphen gebaut.  
Dieses Modul geht tiefer: Wir bauen einen **vollständigen 2-Node-Graphen** mit mehreren State-Feldern,  
verschiedenen Edge-Typen und vollständiger Visualisierung.



**Was behandelt dieses Modul**

| Thema | Inhalt |
|-------|--------|
| **State-Design** | TypedDict mit mehreren Feldern, Reducer vs. Überschreiben |
| **Node-Regeln** | Signatur, partieller Return, Fehlerbehandlung |
| **Edges** | `add_edge()`, kurze Vorschau auf `add_conditional_edges()` |
| **Kompilierung** | `compile()`, `recursion_limit`, Graph-Visualisierung |
| **Ausführung** | `invoke()`, `stream()`, State nach Ausführung inspizieren |
| **LangSmith** | Trace mit `run_name`, Node-by-Node-Sicht |


In [ ]:
#@markdown   <p><font size="4" color='green'>  StateGraph</font> </br></p>



diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart LR
    STATE[/"📦 State\n(zentrale Wahrheit)"/]
    NODE_A["Node A\nliest State\nschreibt State"] --> STATE
    STATE --> NODE_B["Node B\nliest State\nschreibt State"]
    STATE --> EDGE{"Edge /\nRouter"}
    EDGE -->|"Pfad 1"| NODE_A
    EDGE -->|"Pfad 2"| NODE_B

    style STATE fill:#FF9800,color:#fff
    style EDGE  fill:#F44336,color:#fff
    style NODE_A fill:#4CAF50,color:#fff
    style NODE_B fill:#2196F3,color:#fff
'''
mermaid(diagram, width=800)


**Das Praxisbeispiel: Qualitäts-Agent**

Es wird ein **zweistufiger Analyse-Agent** gebaut:

- **Node 1 `entwurf_node`**: Erstellt einen ersten Textentwurf
- **Node 2 `korrektorat_node`**: Verbessert den Entwurf

Jeder Node liest aus dem **gemeinsamen State** und schreibt seine Ergebnisse zurück.

In [ ]:
from langchain.chat_models import init_chat_model
from genai_lib.model_config import JUDGE
llm = init_chat_model(JUDGE)


In [ ]:
#@markdown   <p><font size="4" color='green'>  flowchart</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart LR
    ST([START])
    N1["<b>Node 1</b>\nentwurf_node()\n▸ liest: anfrage\n▸ schreibt: entwurf"]
    N2["<b>Node 2</b>\nkorrektorat_node()\n▸ liest: entwurf\n▸ schreibt: finale_antwort"]
    FIN([FINISH])
    S[/"<b>State</b>\nanfrage: str\nentwurf: str\nfinale_antwort: str\nschritt: int"/]

    ST --> N1 --> N2 --> FIN
    N1 <-.->|"lesen / schreiben"| S
    N2 <-.->|"lesen / schreiben"| S

    style ST fill:#90EE90
    style FIN fill:#FFB6C1
    style S fill:#E8E8FF,stroke:#9999CC
'''

mermaid(diagram, width=900)

# 2 | StateGraph erstellen
---



**Der State: Einzige Wahrheitsquelle**

Der **State** ist das Herzstück jedes LangGraph-Workflows.  
Alle Nodes lesen daraus und schreiben dorthin – sie kommunizieren **nie direkt**.



**TypedDict: Schnell, typsicher, empfohlen**

| Kriterium | TypedDict | Pydantic BaseModel |
|-----------|----------|--------------------|
| **Empfehlung** | ✅ Für Graph-State | ⚠️ Nur für API-Grenzen |
| **Performance** | Kein Overhead | Langsamer (Validation) |
| **Typen** | Statische Hints | Strikte Laufzeit-Validation |



**Reducer: Wie der State aktualisiert wird**

LangGraph unterscheidet zwei Verhaltensweisen bei State-Updates:

| Typ | Verhalten | Beispiel |
|-----|-----------|----------|
| **Überschreiben** (Standard) | Neuer Wert ersetzt alten | `schritt: 2` → `schritt: 3` |
| **Reducer** (`add_messages`) | Neue Werte werden **angehängt** | `[msg1]` + `[msg2]` → `[msg1, msg2]` |

> **Wichtig:** Für `messages` immer `Annotated[list, add_messages]` verwenden –  
> sonst wird der Gesprächsverlauf bei jedem Node-Return überschrieben!

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages

# ─── State-Definition ────────────────────────────────────────────────────
class QualitaetsState(TypedDict):
    """State des Briefing-Qualitäts-Agenten.

    Alle Felder werden von den Nodes geteilt. Der State bleibt bewusst klein,
    aber die Qualitätsprüfung bezieht sich jetzt auf Quellenbindung und
    Korpusgrenze des Meeting- & Research-Briefing-Agenten.
    """
    messages:       Annotated[list, add_messages]  # Reducer: immer anhaengen
    anfrage:        str   # Eingabe des Nutzers (unveraendert)
    entwurf:        str   # Node 1 schreibt den Briefing-Entwurf
    quellen:        list[str]  # Demo-Quellen/Evidenzen für die Pruefung
    finale_antwort: str   # Node 2 schreibt hier
    qualitaetsstatus: str # ok / ueberarbeiten
    risiko:         str   # kurze Begruendung für Review
    schritt:        int   # Zaehlt Ausführungsschritte

# ─── Initialer State ─────────────────────────────────────────────────────
start_state: QualitaetsState = {
    "messages":       [],
    "anfrage":        "",    # Wird beim Aufruf gesetzt
    "entwurf":        "",
    "quellen":        [],
    "finale_antwort": "",
    "qualitaetsstatus": "",
    "risiko":         "",
    "schritt":        0,
}
print("State-Schema:")
for feld, wert in start_state.items():
    print(f"  {feld:18s}: {type(wert).__name__}")

In [ ]:
from langgraph.graph import StateGraph, START, END

# StateGraph mit dem State-Schema initialisieren
builder = StateGraph(QualitaetsState)

print("StateGraph erstellt")
print("Nächste Schritte: Nodes hinzufügen, Edges verbinden, kompilieren")

# 3 | Nodes definieren
---


**Die 4 Regeln für Node-Funktionen**

Eine Node-Funktion empfängt immer den vollständigen State und gibt ausschließlich ein `dict` zurück — also `def node(state: State) -> dict:`. Dabei müssen nur die veränderten Felder zurückgegeben werden; LangGraph merged automatisch mit dem bestehenden State, sodass unveränderte Felder erhalten bleiben. Treten Fehler auf, werden diese nicht mit `raise` weitergeleitet, sondern als String in ein Fehlerfeld des States geschrieben — das hält den Graphen stabil. Schließlich gilt: Nodes sind pure Funktionen ohne globale Seiteneffekte; der State ist die einzige Kommunikationsebene zwischen Nodes.



**Partieller Return – warum kein vollständiger State?**

LangGraph **merged** automatisch:

```python
**Aktueller State:**
**{ anfrage: "Hallo", entwurf: "", schritt: 0 }**

**Node gibt zurück:**
return {"entwurf": "Erster Entwurf...", "schritt": 1}

**Neuer State nach Merge:**
**{ anfrage: "Hallo", entwurf: "Erster Entwurf...", schritt: 1 }**
```

> `anfrage` bleibt erhalten, obwohl der Node es nicht zurückgegeben hat!

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

# ─── Node 1: Briefing-Entwurf erstellen ──────────────────────────────────
def entwurf_node(state: QualitaetsState) -> dict:
    """Erstellt einen ersten Meeting-Briefing-Entwurf mit Demo-Quellen."""
    frage = state["anfrage"]
    prompt = [
        HumanMessage(content=(
            "Erstelle einen knappen Meeting- & Research-Briefing-Entwurf. "
            "Nenne nur belegbare Aussagen und markiere fehlende Evidenz.\n\n"
            f"Frage: {frage}"
        ))
    ]
    response = llm.invoke(prompt, config=run_cfg)
    quellen = state.get("quellen") or ["meeting_briefing_korpus: fachartikel_rag_evaluation.pdf"]
    return {
        "messages": [*prompt, response],
        "entwurf": response.content,
        "quellen": quellen,
        "schritt": state.get("schritt", 0) + 1,
    }

In [ ]:
# ─── Node 2: Qualitätsprüfung durchführen ────────────────────────────────
def korrektorat_node(state: QualitaetsState) -> dict:
    """Prüft den Briefing-Entwurf aus Node 1.

    Liest: state["entwurf"], state["quellen"]
    Schreibt: finale_antwort, qualitaetsstatus, risiko, schritt, messages
    """
    print(f"  [Node 2] Prüfe Briefing-Entwurf ({len(state['entwurf'])} Zeichen)")

    quellen = state.get("quellen", [])
    hat_quellen = bool(quellen)
    status = "ok" if hat_quellen else "ueberarbeiten"
    risiko = "Quellen vorhanden." if hat_quellen else "Keine belegende Quelle im State."
    prompt = [
        HumanMessage(content=(
            "Verbessere den Briefing-Entwurf nur auf Basis der angegebenen Quellen. "
            "Bei fehlender Evidenz schreibe ausdrücklich 'Nicht im Korpus'.\n\n"
            f"Entwurf:\n{state['entwurf']}\n\nQuellen: {quellen}\nQualitätsstatus: {status}\nRisiko: {risiko}"
        ))
    ]
    response = llm.invoke(prompt, config=run_cfg)

    return {
        "messages": [*prompt, response],
        "finale_antwort": response.content,
        "qualitaetsstatus": status,
        "risiko": risiko,
        "schritt": state["schritt"] + 1,
    }

In [ ]:
#@markdown   <p><font size="4" color='green'>  flowchart</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart TB
    subgraph Node1["entwurf_node() – Node 1"]
        N1R["Liest: anfrage"] --> N1L["LLM: Erstellt Entwurf"]
        N1L --> N1W["Schreibt: entwurf, schritt"]
    end

    subgraph State["Gemeinsamer State"]
        direction LR
        F1["anfrage: str"]
        F2["entwurf: str"]
        F3["finale_antwort: str"]
        F4["schritt: int"]
        F5["messages: list"]
    end

    subgraph Node2["korrektorat_node() – Node 2"]
        N2R["Liest: entwurf"] --> N2L["LLM: Verbessert Text"]
        N2L --> N2W["Schreibt: finale_antwort, schritt"]
    end

    Node1 <-->|merge| State
    Node2 <-->|merge| State

    style State fill:#E8E8FF,stroke:#9999CC
    style Node1 fill:#E8FFE8
    style Node2 fill:#FFE8E8
'''

mermaid(diagram, width=1050)

# 4 | Edges verbinden
---



**Edge-Typen**

| Typ | Methode | Wann verwenden |
|-----|---------|----------------|
| **Einfache Edge** | `add_edge(from, to)` | Immer dieser Weg |
| **Conditional Edge** | `add_conditional_edges(from, fn)` | Entscheidung zur Laufzeit |

**START und END**

```python
from langgraph.graph import START, END

**START = spezieller Eingangsknoten (kein Node, kein Code)**
**END   = spezieller Ausgangsknoten (Workflow beenden)**
builder.add_edge(START, "mein_node")   # Einstieg
builder.add_edge("mein_node", END)     # Ausstieg
```

**Conditional Edges – kurze Vorschau (Details folgen im nächsten Modul)**

```python
**Routing-Funktion: gibt den Namen des nächsten Nodes zurück**
def routing_fn(state: State) -> str:
    if state["Qualität"] == "schlecht":
        return "nochmal_verbessern"   # Schleife
    return END                         # Fertig

builder.add_conditional_edges(
    "korrektorat",    # Von diesem Node
    routing_fn,       # Diese Funktion entscheidet
)
```

> *Conditional Routing & Tool-Loop* zeigt Conditional Edges und Tool-Loops im Detail.

In [ ]:
# Nodes zum Builder hinzufügen
builder.add_node("entwurf",    entwurf_node)
builder.add_node("korrektorat", korrektorat_node)

# Edges verbinden: START → entwurf → korrektorat → END
builder.add_edge(START,         "entwurf")
builder.add_edge("entwurf",     "korrektorat")
builder.add_edge("korrektorat", END)

print("Nodes registriert:", list(builder.nodes.keys()))
print("Graph-Struktur:")
print("  START → entwurf → korrektorat → END")

# 5 | Graph kompilieren & testen
---



**Kompilierung: Builder → unveränderlicher Graph**

Nach `compile()` kann der Graph **nicht mehr verändert** werden.  
Die Kompilierung prüft den Graphen auf Vollständigkeit und erzeugt den ausführbaren Workflow.

| Option | Beschreibung | Standard |
|--------|-------------|----------|
| `checkpointer=` | Persistenz (InMemorySaver/PostgresSaver) | Kein Checkpointing |
| `interrupt_before=` | Nodes bei denen pausiert wird | `[]` |
| `interrupt_after=` | Nodes nach denen pausiert wird | `[]` |

> **Best Practice:** Immer `draw_mermaid_png()` nach `compile()` aufrufen –  
> zeigt den **tatsächlich** kompilierten Graphen.

In [ ]:
# Graph kompilieren
graph = builder.compile()

In [ ]:
# Graph visualisieren
from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
# Graph ausführen mit invoke()
initial_state: QualitaetsState = {
    "messages":       [],
    "anfrage":        "Die Bedeutung von RAG-Systemen in modernen KI-Anwendungen",
    "entwurf":        "",
    "finale_antwort": "",
    "schritt":        0,
}

print("Starte Graphen...\n")
ergebnis = graph.invoke(initial_state, config=run_cfg)

mprint(f"""
## Ergebnis nach invoke()

**Schritte ausgeführt:** {ergebnis["schritt"]}

**Entwurf (Node 1):**
{ergebnis["entwurf"]}

---

**Finale Antwort (Node 2):**
{ergebnis["finale_antwort"]}

**Nachrichten im State:** {len(ergebnis["messages"])} Einträge
""")

In [ ]:
# State nach Ausführung inspizieren
print("=== State-Inspektion ===")
for feld, wert in ergebnis.items():
    if feld == "messages":
        print(f"  messages: {len(wert)} Einträge")
        for i, msg in enumerate(wert):
            typ = type(msg).__name__
            inhalt = msg.content[:60].replace("\n", " ") + "..."
            print(f"    [{i}] {typ}: {inhalt}")
    elif isinstance(wert, str) and len(wert) > 60:
        print(f"  {feld}: {wert[:60]}...")
    else:
        print(f"  {feld}: {repr(wert)}")

In [ ]:
# Graph mit stream() Schritt für Schritt verfolgen
print("=== stream() - Node-by-Node ===\n")

stream_state: QualitaetsState = {
    **initial_state,
    "anfrage": "Warum ist Checkpointing in LangGraph wichtig?",
}

for schritt, event in enumerate(graph.stream(stream_state, stream_mode="updates")):
    node_name = list(event.keys())[0]
    node_output = event[node_name]

    print(f"--- Schritt {schritt + 1}: Node '{node_name}' ---")
    for feld, wert in node_output.items():
        if feld == "messages":
            print(f"  messages: +{len(wert)} neue Einträge")
        elif isinstance(wert, str) and len(wert) > 80:
            print(f"  {feld}: {wert[:80]}...")
        else:
            print(f"  {feld}: {repr(wert)}")
    print()

# 6 | Graph im LangSmith
---



LangGraph-Traces in LangSmith zeigen eine **Node-by-Node-Sicht** des Workflows –
ideal zum Verstehen und Debuggen.

**Was LangSmith für StateGraphen zeigt**

| Ebene | Inhalt | Nutzen |
|-------|--------|--------|
| **Graph-Ebene** | Gesamtlaufzeit, Input/Output | Überblick |
| **Node-Ebene** | Ein Eintrag pro Node | Welcher Node lief wie lang? |
| **LLM-Ebene** | Prompt, Response, Tokens | Prompt-Debugging |
| **State-Diff** | State vor/nach jedem Node | Was hat sich verändert? |

**run_name und Tags**

```python
config = {
    "run_name": "Qualitäts-Agent",
    "tags":     ["m09", "zwei-node-graph"],
}
graph.invoke(state, config=config)
```

In [ ]:
# Ausführung mit LangSmith-Konfiguration
langsmith_config = {
    "run_name": "M09_StateGraph_Basics",
    "tags":     ["m09", "zwei-node-graph", "invoke"],
    "metadata": {"modul": "M09", "version": "1.0"},
}

trace_state: QualitaetsState = {
    "messages":       [],
    "anfrage":        "Erkläre den Unterschied zwischen invoke() und stream() in LangGraph",
    "entwurf":        "",
    "finale_antwort": "",
    "schritt":        0,
}

print("Starte Trace...\n")
trace_result = graph.invoke(trace_state, config=langsmith_config)

mprint(f"""
## LangSmith-Trace

**Projekt:** M09-StateGraph-Basics
**Run-Name:** Qualitäts-Agent-Demo
**Schritte:** {trace_result["schritt"]}

**Finale Antwort:**
{trace_result["finale_antwort"][:400]}...

> Trace in LangSmith unter Projekt **M09-StateGraph-Basics** einsehen.
""")

**Was passiert hier?**

1. `StateGraph(...)` — definiert den Graphen mit dem State-Schema

In [ ]:
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M09-StateGraph-Basics", limit=3, show_steps=True)

# 7 | Ausblick: Agent-Loop mit LangGraph
---


Das frühere Kapitel 6 aus M14 gehört fachlich hierher: Sobald ein Agent nicht nur eine einzelne RAG-Antwort erzeugt, sondern einen kontrollierbaren Loop mit State, Memory oder HITL braucht, ist LangGraph der passende Ort.

In M09 bleibt dieser Abschnitt ein **Architektur-Ausblick**. Die RAG-spezifische Umsetzung wird später in M14/M17/M20 wieder aufgegriffen.

| Frage | Einfache Agenten-API | LangGraph-Variante |
|---|---|---|
| Wer steuert den Ablauf? | Verborgen in `create_agent()` | Explizite Nodes und Edges |
| Wo liegt der Zustand? | Intern | Im State-Schema |
| Wie wird kontrolliert? | Schwer sichtbar | Conditional Edges, Interrupts, Checkpointer |
| Warum wichtig für Mara? | Schnelle Demo | Nachvollziehbare Briefing-Agent-Architektur |

**Merksatz:** M14 zeigt den RAG-Agenten als nutzbare Anwendung; M09 erklärt, warum derselbe Ablauf als Graph kontrollierbarer wird.


# A | Aufgaben
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabenstellungen unten bieten Anregungen; alternative Herausforderungen sind möglich.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs darf und soll generative KI auch als Unterstützung beim Lernen und Entwickeln genutzt werden. Geeignet ist sie zum Beispiel, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.


<p><font color='black' size="5">
Drei-Node-Research-Graph
</font></p>

Einen StateGraph für den Meeting-Briefing-Agent bauen. Der Graph analysiert eine Frage, erstellt einen Antwortentwurf und prüft die Qualität.


**Grundlagen**
1. Einen einfachen State mit den Feldern `frage`, `analyse`, `antwort` und `status` definieren.
2. Einen linearen Graphen mit mindestens 2 Nodes bauen.
3. Den Graph einmal mit `invoke()` testen.

**✅ Erledigt wenn:** `mein_graph.invoke(...)` gibt einen State mit Analyse und Antwort zurück; der Selbstcheck läuft ohne `AssertionError`.


In [ ]:
# Grundlagen: StateGraph mit mindestens 2 Nodes
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class ResearchGraphState(TypedDict):
    frage: str
    analyse: str
    antwort: str
    status: str

def frage_analysieren(state: ResearchGraphState) -> dict:
    frage = state.get("frage", "")
    if "rag" in frage.lower() or "retrieval" in frage.lower():
        analyse = "Korpusnahe Briefing-Frage erkannt."
    else:
        analyse = "Korpusbezug unklar; vorsichtige Antwort nötig."
    return {"analyse": analyse, "status": "analysiert"}

def antwort_entwerfen(state: ResearchGraphState) -> dict:
    return {
        "antwort": "Entwurf auf Basis der Analyse: " + state.get("analyse", ""),
        "status": "entworfen",
    }

builder = StateGraph(ResearchGraphState)
builder.add_node("frage_analysieren", frage_analysieren)
builder.add_node("antwort_entwerfen", antwort_entwerfen)
builder.add_edge(START, "frage_analysieren")
builder.add_edge("frage_analysieren", "antwort_entwerfen")
builder.add_edge("antwort_entwerfen", END)
mein_graph = builder.compile()

grundlagen_ergebnis = mein_graph.invoke({
    "frage": "Warum verbessert RAG die Zuverlässigkeit?",
    "analyse": "",
    "antwort": "",
    "status": "neu",
}, config=run_cfg)
print("Graph kompiliert:", list(mein_graph.nodes))
print("Testergebnis:", grundlagen_ergebnis)


In [ ]:
# ✅ Selbstcheck Grundlagen
# Voraussetzung: Graph in 'mein_graph', Ergebnis in 'grundlagen_ergebnis'.

assert 'mein_graph' in dir() or 'mein_graph' in locals(),     "❌ Variable 'mein_graph' fehlt - Graph kompilieren und speichern"
assert hasattr(mein_graph, "invoke"),     "❌ Graph hat kein invoke() - wurde builder.compile() aufgerufen?"
assert 'grundlagen_ergebnis' in dir() or 'grundlagen_ergebnis' in locals(),     "❌ grundlagen_ergebnis fehlt - Graph mit invoke() testen"
assert len(grundlagen_ergebnis.get("analyse", "")) > 10,     "❌ Analyse fehlt oder ist zu kurz"
assert len(grundlagen_ergebnis.get("antwort", "")) > 10,     "❌ Antwort fehlt oder ist zu kurz"

print("✅ Grundlagen-Selbstcheck bestanden!")


**Aufbau**
1. Einen dritten Node `qualitaets_check` ergänzen.
2. Nodes sequentiell verbinden: START -> Analyse -> Entwurf -> Qualität -> END.
3. Den Graph kompilieren und optional mit `draw_mermaid_png()` visualisieren.
4. Einen Test mit einer Briefing-Frage durchführen.

**✅ Erledigt wenn:** Der Graph hat mindestens 3 Nodes, das Ergebnis enthält einen Qualitätsstatus und der Selbstcheck läuft fehlerfrei.


In [ ]:
# Aufbau: 3-Node-Research-Graph
def qualitaets_check(state: ResearchGraphState) -> dict:
    antwort = state.get("antwort", "")
    if "Korpusnahe" in state.get("analyse", ""):
        status = "Quellenhinweis erforderlich"
    else:
        status = "Unsicherheit markieren"
    return {"antwort": antwort + " | Qualitätsstatus: " + status, "status": status}

builder3 = StateGraph(ResearchGraphState)
builder3.add_node("frage_analysieren", frage_analysieren)
builder3.add_node("antwort_entwerfen", antwort_entwerfen)
builder3.add_node("qualitaets_check", qualitaets_check)
builder3.add_edge(START, "frage_analysieren")
builder3.add_edge("frage_analysieren", "antwort_entwerfen")
builder3.add_edge("antwort_entwerfen", "qualitaets_check")
builder3.add_edge("qualitaets_check", END)
mein_graph = builder3.compile()

aufbau_ergebnis = mein_graph.invoke({
    "frage": "Wie sollte eine RAG-Antwort belegt werden?",
    "analyse": "",
    "antwort": "",
    "status": "neu",
}, config=run_cfg)
print("Research-Graph kompiliert:", list(mein_graph.nodes))
print("Aufbau-Ergebnis:", aufbau_ergebnis)
# Optional in Colab:
# from IPython.display import Image, display
# display(Image(mein_graph.get_graph().draw_mermaid_png()))


In [ ]:
# ✅ Selbstcheck Aufbau
# Voraussetzung: 3-Node-Graph in 'mein_graph', Ergebnis in 'aufbau_ergebnis'.

assert 'mein_graph' in dir() or 'mein_graph' in locals(),     "❌ Variable 'mein_graph' fehlt"
assert hasattr(mein_graph, "invoke"),     "❌ Graph hat kein invoke()"
assert len(list(mein_graph.nodes)) >= 3,     "❌ Weniger als 3 Nodes - Qualitätsnode ergänzen"
assert 'aufbau_ergebnis' in dir() or 'aufbau_ergebnis' in locals(),     "❌ aufbau_ergebnis fehlt"
assert "Qualitätsstatus" in aufbau_ergebnis.get("antwort", ""),     "❌ Qualitätsstatus fehlt in der Antwort"

print("✅ Aufbau-Selbstcheck bestanden!")


**Vertiefung**
1. `stream()` statt `invoke()` verwenden und den State nach jedem Node einzeln ausgeben.
2. In einer Zeile dokumentieren, was sich am Ausgabe-Verhalten ändert.
3. LangSmith-Tracing aktiv lassen und prüfen, welche Nodes im Trace sichtbar sind.
4. Optional ein Feld `risiko: str` ergänzen und im Qualitätsnode setzen.

**✅ Erledigt wenn:** `stream_ergebnisse` enthält Updates nach jedem Node; `stream_beobachtung` beschreibt den Unterschied zu `invoke()`.


In [ ]:
# Vertiefung: stream() statt invoke()
stream_ergebnisse = []
anfangs_state = {
    "frage": "Warum braucht ein Meeting-Briefing-Agent Quellenbindung?",
    "analyse": "",
    "antwort": "",
    "status": "neu",
}
for update in mein_graph.stream(anfangs_state, stream_mode="updates"):
    stream_ergebnisse.append(update)
    print("Update nach Node:", list(update.keys()))

stream_beobachtung = (
    "stream() liefert nach jedem Node ein Update; invoke() gibt erst nach dem vollständigen Durchlauf den finalen State zurück."
)
print("Beobachtung:", stream_beobachtung)


In [ ]:
# ✅ Selbstcheck Vertiefung
# Voraussetzung: stream_ergebnisse und stream_beobachtung wurden gespeichert.

import os
assert os.environ.get("LANGSMITH_TRACING") == "true",     "❌ LangSmith-Tracing nicht aktiv - LANGSMITH_TRACING muss 'true' sein"
assert 'stream_ergebnisse' in dir() or 'stream_ergebnisse' in locals(),     "❌ stream_ergebnisse fehlt"
assert isinstance(stream_ergebnisse, list) and len(stream_ergebnisse) >= 2,     "❌ stream_ergebnisse enthält zu wenige Updates"
assert 'stream_beobachtung' in dir() or 'stream_beobachtung' in locals(),     "❌ stream_beobachtung fehlt"
assert isinstance(stream_beobachtung, str) and len(stream_beobachtung) >= 30,     "❌ stream_beobachtung ist zu kurz"

print("✅ Vertiefung-Selbstcheck bestanden!")


**Was passiert hier?**

1. `compile(...)` — schließt den Graphen ab und erzeugt das ausführbare Objekt

<p><font color='darkblue' size="4">
 <b>Viz</b>
</font></p>

- [LangGraph](https://editor.p5js.org/ralf.bendig.rb/full/EUzaFq4C4)
- [KI-Agenten-Architektur](https://editor.p5js.org/ralf.bendig.rb/full/Viso2emNI)


# B | Dokumente zum Weiterlesen
---

Ergänzende Artikel aus der Kurs-Dokumentation:

- [State Management](https://ralf-42.github.io/Agenten/04-agenten-implementierung/ablauf-zustand/state-management.html)
- [Einsteiger LangGraph](https://ralf-42.github.io/Agenten/05-frameworks/einsteiger-langgraph.html)
- [LangGraph Best Practices](https://ralf-42.github.io/Agenten/05-frameworks/langgraph-best-practices.html)
- [LangChain/LangGraph Cheatsheet](https://ralf-42.github.io/Agenten/05-frameworks/langchain-langgraph-cheatsheet.html)
- [Checkliste Agentensystem](https://ralf-42.github.io/Agenten/04-agenten-implementierung/checkliste-agentensystem.html)
- [Code Standards](https://ralf-42.github.io/Agenten/10-ressourcen/standards.html)

